# ML Experimentation with Real-Time Inference & Model Monitoring

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong> This notebook has been tested using <strong>SageMaker Distribution Image 4.6.0</strong> and the <strong>SageMaker Python SDK version 3.27.0</strong>.
</div>

## Overview
This notebook demonstrates an end-to-end machine learning workflow with [Amazon SageMaker AI](https://docs.aws.amazon.com/sagemaker/latest/dg/whatis.html) that includes:

- **Model Registry**: Deploy the latest **approved** model version produced by the SageMaker pipeline in [notebook 03](03-sagemaker-pipeline.ipynb) — no retraining
- **MLflow Integration**: Track experiments and log artifacts to [SageMaker AI MLflow App](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html)
- **Real-Time Endpoint**: Deploy a SageMaker [real-time endpoint](https://docs.aws.amazon.com/sagemaker/latest/dg/realtime-endpoints.html) with [data capture](https://docs.aws.amazon.com/sagemaker/latest/dg/model-monitor-data-capture.html) enabled
- **Data Drift Monitoring**: Use [Evidently (open-source)](https://www.evidentlyai.com/evidently-oss) to detect data drift on captured data
- **Model Quality Monitoring**: Use Evidently (open-source) to generate a model quality report
- **Drift Metrics in MLflow**: Log drift metrics and reports to MLflow for tracking

||||
|---|---|---|
|1. |Experiment in a notebook ||
|2. |Scale with SageMaker AI processing jobs and SageMaker SDK ||
|3. |Operationalize with ML pipeline, model registry, and feature store ||
|4. |Add model and data monitoring |**<<<< YOU ARE HERE**|
|5. |Add a model building CI/CD pipeline ||
|6. |Add a model deployment pipeline ||

## Workflow
The diagram below shows the full monitoring workflow using a real-time inference endpoint.

1. Take the approved model version from the [SageMaker Model Registry](https://docs.aws.amazon.com/sagemaker/latest/dg/model-registry.html), together with the baseline data set the pipeline stored in S3.
2. Deploy the model in a real-time endpoint with data capture enabled; data capture stores the input and output from the endpoint in S3.
3. Use Evidently to calculate data drift and model quality based on the baseline dataset and the data captured from the endpoint.
    1. In the diagram, the Evidently code runs in a [processing job](https://docs.aws.amazon.com/sagemaker/latest/dg/processing-job.html) inside a [pipeline](https://docs.aws.amazon.com/sagemaker/latest/dg/pipelines.html). This enables running the drift calculations based on a schedule or based on a trigger from S3. For simplicity, this notebook runs the Evidently calculations locally.
4. Save the data drift and model quality reports to MLflow for tracking and comparison.
5. Optionally, trigger an alert if data or model drift exceeds a certain threshold. The alert can be raised directly in Slack through [Amazon SNS](https://docs.aws.amazon.com/sns/latest/dg/welcome.html).

![Model monitoring workflow with real-time endpoints](img/arch-sagemaker-inference-predictiveml-monitoring-RealTime-inf.png "Model monitoring workflow with real-time endpoints")

### Business Problem
This sample uses the [Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing) dataset from the UCI Machine Learning Repository. It can be used to predict whether a bank customer will subscribe to a term deposit based on:
- Demographics (age, job, marital status, education)
- Financial information (credit default, housing loan, personal loan)
- Campaign data (contact type, month, day of week, duration)
- Economic indicators (employment rate, consumer price index, etc.)

---

<div style="border: 4px solid #d32f2f; border-radius: 10px; padding: 20px; background-color: #ffebee; margin: 15px 0;">

### ⛔ IMPORTANT: Python 3.12 Required

All notebooks in this workshop **must** run on **Python 3.12**. The cell below will fail with an `AssertionError` if you are using a different Python version.

If you see an error, follow these steps:

1. Open a **Terminal** in JupyterLab (**File → New → Terminal**)
2. Run the setup script:
   ```bash
   cd ~/amazon-sagemaker-from-idea-to-production
   ./setup-py312.sh
   ```
3. **Refresh your browser** to pick up the new kernel
4. In this notebook, go to **Kernel → Change Kernel → Python 3.12**
5. **Restart the kernel** and re-run from the top

</div>

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;"> 
<h3>⚠️ SIMULATED PRODUCTION DATA WITH ARTIFICIAL DRIFT</h3>
<p><strong>Purpose:</strong> Use the notebook to demonstrate drift detection capabilities, not for production deployments</p>

In [ ]:
import sys

assert sys.version_info[:2] == (3, 12), f"Expected Python 3.12, got {sys.version_info.major}.{sys.version_info.minor}. Please make sure you run on Python 3.12"

## 1: Prerequisites & Setup

Install required packages and import libraries.

In [ ]:
%store -r 

%store

try:
    initialized
except NameError:
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN 00-start-here notebook   ")
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")

In [ ]:
import boto3
import sagemaker
import mlflow
import pandas as pd
import numpy as np
import os
import json
import time
import importlib.util
from datetime import datetime
from sklearn.metrics import roc_auc_score

from sagemaker.core.helper.session_helper import Session
from sagemaker.core.common_utils import name_from_base
from sagemaker.core.resources import Model, EndpointConfig, Endpoint, ModelPackage
from sagemaker.core.shapes import (
    ContainerDefinition, ProductionVariant,
    DataCaptureConfig, CaptureOption, CaptureContentTypeHeader,
)

from evidently import Report, Dataset, DataDefinition, BinaryClassification
from evidently.presets import DataDriftPreset, DataSummaryPreset, ClassificationPreset
from evidently.metrics import *

# Same feature engineering the pipeline's preprocess step uses
from pipeline_steps.preprocess import engineer_features

print(f'MLflow version: {mlflow.__version__}')

In [ ]:
sagemaker_session = Session()

print('AWS Configuration:')
print(f'  Region: {region}')
print(f'  S3 Bucket: {bucket_name}')
print(f'  IAM Role: {sm_role}')
print(f'  Data Prefix: {bucket_prefix}')

---

## 2: SageMaker AI MLflow App Configuration

### What is a SageMaker AI MLflow App?

A fully managed service based on open-source MLflow that provides:
- **Experiment Tracking**: Log parameters, metrics, and artifacts
- **Model Registry**: Version and manage models
- **SageMaker Integration**: Automatic registration to SageMaker AI Model Registry
- **Reproducibility**: Track code versions and dependencies
- **Collaboration**: Share experiments across teams

### Instructions

1. In SageMaker Studio, navigate to the left sidebar
2. Click on "MLflow" panel under "Applications"
3. Find your MLflow app (usually named "DefaultMLFlowApp") or create one if necessary
4. Copy the app name and edit it in the cell below if you need to use a specific app

The cell below checks if the MLflow App exists and will fetch the corresponding ARN.

The ARN fetched by the previous cell is set as the default MLflow tracking URI. A new experiment is created within MLflow if required, otherwise the existing experiment is set as the default.

In [ ]:
mlflow.set_tracking_uri(mlflow_arn)
mlflow_experiment_name = 'ml-xgb-train-monitor'

try:
    # Try to create a new experiment
    experiment_id = mlflow.create_experiment(mlflow_experiment_name)
    print(f'Created new MLflow experiment: {mlflow_experiment_name}')
    print(f'Experiment ID: {experiment_id}')
except:
    # Experiment already exists, set it as active
    mlflow.set_experiment(mlflow_experiment_name)
    experiment = mlflow.get_experiment_by_name(mlflow_experiment_name)
    print(f'Using existing MLflow experiment: {mlflow_experiment_name}')
    print(f'Experiment ID: {experiment.experiment_id}')

---

## 3: Load the pipeline's data sets

The model you deploy was trained by the SageMaker pipeline in [notebook 03](03-sagemaker-pipeline.ipynb). To monitor it, you need data in exactly the format that model was trained on, so this notebook reuses the pipeline's outputs instead of preparing its own:

- `test/test_x.csv` and `test/test_y.csv`: the held-out test features and labels, which play the role of production traffic and ground truth
- `baseline/baseline.csv`: the full feature table, used as the reference distribution for drift detection

The pipeline writes these files without a header. The feature names come from `engineer_features()` in [`pipeline_steps/preprocess.py`](pipeline_steps/preprocess.py), the same function the pipeline's preprocess step runs.

In [ ]:
raw_df = pd.read_csv('data/bank-additional/bank-additional-full.csv', sep=';')
feature_names = engineer_features(raw_df).columns.drop('y').tolist()

X_test = pd.read_csv(f'{test_s3_url}/test_x.csv', header=None, names=feature_names)
y_test = pd.read_csv(f'{test_s3_url}/test_y.csv', header=None).iloc[:, 0].astype(int)
baseline_df = pd.read_csv(f'{baseline_s3_url}/baseline.csv', header=None, names=feature_names)

assert len(X_test) == len(y_test), 'test_x and test_y have different lengths'
print(f'Features:      {len(feature_names)}')
print(f'Test set:      {X_test.shape}  (positive rate {y_test.mean():.3f})')
print(f'Baseline set:  {baseline_df.shape}')
X_test.head()

Save the baseline **with a header** and upload it. The Evidently reports and the monitoring Processing Job later in this notebook read it by column name.

In [ ]:
os.makedirs('data', exist_ok=True)
baseline_df.to_csv('data/baseline.csv', index=False, header=True)
baseline_s3 = sagemaker_session.upload_data('data/baseline.csv', bucket_name, f'{bucket_prefix}/data/baseline')
print(f'Baseline S3: {baseline_s3}')

---

## 4: Select the approved model from the Model Registry

The pipeline's register step logs an **inference specification** on the MLflow model before registering it. The Model Package that is auto-synced to the SageMaker Model Registry is therefore deployable as-is: it carries the serving image, the model data location, and the environment.

The cell below picks the model version to monitor from the model package group created by the pipeline:

1. the most recent version that is already `Approved` (for example by the CI/CD flow in [notebook 06](06-deploy.ipynb) on a re-run of the workshop), otherwise
2. the most recent `PendingManualApproval` version, which you approve in the next step.

Only versions with an `InferenceSpecification` qualify; older versions registered without one can't be deployed from the registry.

In [ ]:
sm_client = boto3.client('sagemaker')

# Model package group written by the pipeline (notebook 03), restored via %store
try:
    model_package_group = target_model_package_group_name
except NameError:
    try:
        model_package_group = model_package_group_name
    except NameError:
        model_package_group = registered_model_name
print(f'Model package group: {model_package_group}')

def list_packages(status):
    return [
        p for page in sm_client.get_paginator('list_model_packages').paginate(
            ModelPackageGroupName=model_package_group,
            ModelApprovalStatus=status,
            SortBy='CreationTime',
            SortOrder='Descending',
        )
        for p in page['ModelPackageSummaryList']
    ]

def is_deployable(arn):
    return 'InferenceSpecification' in sm_client.describe_model_package(ModelPackageName=arn)

model_package_arn = next((p['ModelPackageArn'] for p in list_packages('Approved') if is_deployable(p['ModelPackageArn'])), None)
needs_approval = model_package_arn is None
if needs_approval:
    model_package_arn = next((p['ModelPackageArn'] for p in list_packages('PendingManualApproval') if is_deployable(p['ModelPackageArn'])), None)

if model_package_arn is None:
    raise RuntimeError(
        f'No deployable Approved or PendingManualApproval model version in {model_package_group}. '
        'Run the pipeline in notebook 03 first.'
    )

described = sm_client.describe_model_package(ModelPackageName=model_package_arn)
metadata = described.get('CustomerMetadataProperties', {})
print(f'Selected model package: {model_package_arn}')
print(f'  approval status:      {described.get("ModelApprovalStatus")}')
print(f'  image:                {described["InferenceSpecification"]["Containers"][0]["Image"]}')
print(f'  MLflow model id:      {metadata.get("MlflowModelId", "n/a")}')

### Make sure the model version is approved

If the selected version is still `PendingManualApproval`, approve it, as in [notebook 02](02-sagemaker-containers.ipynb).

<div class="alert alert-warning">
If you already created the MLOps project from notebooks 05 and 06 (for example on a re-run of the workshop), approving a model version also triggers its EventBridge rule, which starts the model deployment CI/CD pipeline.
</div>

In [ ]:
model_package = ModelPackage.get(model_package_name=model_package_arn)
# DescribeModelPackage does not return ModelPackageName for versioned packages,
# which breaks refresh()/update(). The API accepts an ARN in that field.
model_package.model_package_name = model_package.model_package_arn

if model_package.model_approval_status != 'Approved':
    model_package.update(model_approval_status='Approved')
    model_package.refresh()

assert model_package.model_approval_status == 'Approved', model_package.model_approval_status
print(f'Approval status: {model_package.model_approval_status}')

### Validate the real inference code in this kernel

Before you deploy, check that the model and its serving code work on the test data, as in [notebook 02](02-sagemaker-containers.ipynb). This downloads the same MLflow artifacts the endpoint will download, imports the logged `code/inference.py`, and drives its handlers. It also confirms the test data has the number of features the model expects.

In [ ]:
model_id = metadata.get('MlflowModelId')
if not model_id:
    print('[WARN] Model package has no MlflowModelId metadata, skipping local validation.')
else:
    local_model_dir = mlflow.artifacts.download_artifacts(artifact_uri=f'models:/{model_id}')
    spec = importlib.util.spec_from_file_location(
        'user_inference_module', os.path.join(local_model_dir, 'code', 'inference.py')
    )
    inference_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(inference_module)
    local_model = inference_module.model_fn(local_model_dir)

    assert local_model.num_features() == X_test.shape[1], (
        f'Model expects {local_model.num_features()} features, test data has {X_test.shape[1]}'
    )

    payload = '\n'.join(','.join(map(str, row)) for row in X_test.values).encode('utf-8')
    data = inference_module.input_fn(payload, 'text/csv')
    body, _ = inference_module.output_fn(inference_module.predict_fn(data, local_model), 'text/csv')
    local_predictions = np.array(body.strip().split('\n'), dtype=float)
    print(f'Local test AUC: {roc_auc_score(y_test, local_predictions):.4f}')

## 5: Real-Time Endpoint with Data Capture

Deploy the trained model as a SageMaker real-time endpoint with data capture enabled. Data capture logs the inputs to your endpoint and the inference outputs from your deployed model to Amazon S3. 

### Why Real-Time Endpoints with Data Capture?
- **Low latency**: Sub-second inference for real-time applications
- **Data Capture**: Automatically logs request/response payloads to S3
- **Monitoring**: Captured data enables drift detection and model quality monitoring
- **Production-ready**: Supports auto-scaling, A/B testing, and shadow deployments

### Deploy the approved Model Package version

Deploying from the registry needs no image, script, or artifact URI at the call site. All of that is in the package's `InferenceSpecification`, so a `ContainerDefinition` that names the Model Package is enough. This is the same approach as [notebook 02](02-sagemaker-containers.ipynb), with one addition: the `EndpointConfig` enables **data capture**.

Data capture can save the input, the output, or both, and can sample a subset of requests. Here it captures 100% of inputs and outputs.

<div class="alert alert-info">
This endpoint has no authentication of its own. Access is controlled entirely by IAM (<code>sagemaker:InvokeEndpoint</code>), and the endpoint isn't exposed to the public internet.
</div>

In [ ]:
inference_instance_type = 'ml.m5.xlarge'
endpoint_name = name_from_base("from-idea-to-prod-monitor-ep-", short=True)
model_name = endpoint_name
capture_s3_uri = f's3://{bucket_name}/{bucket_prefix}/data-capture'

Model.create(
    model_name=model_name,
    primary_container=ContainerDefinition(model_package_name=model_package_arn),
    execution_role_arn=sm_role,
)

EndpointConfig.create(
    endpoint_config_name=endpoint_name,
    production_variants=[
        ProductionVariant(
            variant_name='AllTraffic',
            model_name=model_name,
            initial_instance_count=1,
            instance_type=inference_instance_type,
        )
    ],
    data_capture_config=DataCaptureConfig(
        enable_capture=True,
        initial_sampling_percentage=100,
        destination_s3_uri=capture_s3_uri,
        capture_options=[CaptureOption(capture_mode='Input'), CaptureOption(capture_mode='Output')],
        capture_content_type_header=CaptureContentTypeHeader(csv_content_types=['text/csv']),
    ),
)

Endpoint.create(endpoint_name=endpoint_name, endpoint_config_name=endpoint_name)

print(f'Creating endpoint {endpoint_name} from {model_package_arn}')
print(f'Data capture destination: {capture_s3_uri}')
print('Poll its status in the next cell (this takes a few minutes).')

In [ ]:
# Endpoint.create() returns as soon as the request is accepted, so poll until the
# endpoint reaches a terminal state.

sagemaker_client = boto3.client('sagemaker')
print(f'Polling endpoint: {endpoint_name}')

terminal_states = {'InService', 'Failed'}
while True:
    desc = sagemaker_client.describe_endpoint(EndpointName=endpoint_name)
    status = desc['EndpointStatus']
    print(f"  {time.strftime('%H:%M:%S')} | status={status}")
    if status in terminal_states:
        break
    time.sleep(30)

if status != 'InService':
    raise RuntimeError(
        f"Endpoint {endpoint_name} deployment ended with status '{status}': "
        f"{desc.get('FailureReason')}"
    )
print(f'Endpoint {endpoint_name} is InService.')

### Prepare Ground Truth Data & Invoke Endpoint

To calculate model quality (i.e. accuracy, F1 score), ground truth labels must be available to compare to the predicted labels. In production, the ground truth labels are normally collected asynchronously. For example, when a model is used to forecast sales for the next month, the ground truth sales data will only be available after the month has passed. When a model is used to generate recommendations for a retail website, the ground truth might be available within minutes when a user decides to purchase the product or not. The process for collecting ground truth labels depends on the ML use case.

In this sample notebook, the test dataset is used to calculate model quality so the ground truth labels are already available. To replicate a production use case, the ground truth labels are saved separately, to be merged with the inference results later. Each inference input is assigned a unique ID (e.g. `inf-000010`) to enable merging.

In [ ]:
test_features = X_test.copy()
rng = np.random.default_rng(42)

# SIMULATED PRODUCTION DRIFT: apply shifts here so the Data Capture actually
# contains drifted records (the Processing Job compares capture vs baseline).
# Do NOT do this in production — this is purely to demonstrate drift detection.

# 1. More contacts per campaign: `campaign` is min-max scaled to [0, 1]
test_features['campaign'] = (
    test_features['campaign'] * 1.5 + rng.normal(0, 0.02, len(test_features))
).clip(0, 1)

# 2. An older customer base: move 70% of the records up one age bucket
age_cols = [c for c in test_features.columns if c.startswith('age_')]
bucket = test_features[age_cols].values.argmax(axis=1)
bucket = np.minimum(bucket + (rng.random(len(test_features)) < 0.7), len(age_cols) - 1)
test_features[age_cols] = np.eye(len(age_cols), dtype=int)[bucket]

ground_truth = {f'inf-{idx:06d}': int(label) for idx, label in enumerate(y_test)}
gt_df = pd.DataFrame(list(ground_truth.items()), columns=['inference_id', 'target'])
gt_df.to_csv('data/ground_truth.csv', index=False)
print(f'Ground truth stored for {len(gt_df)} records')

Now the test dataset, without the ground truth labels, can be sent to the real-time endpoint for inference.

In [ ]:
runtime_client = boto3.client('sagemaker-runtime', region_name=region)

print(f'Sending {len(test_features)} requests to endpoint...')
for idx, (i, row) in enumerate(test_features.iterrows()):
    inference_id = f'inf-{idx:06d}'
    payload = ','.join(str(v) for v in row.values)
    response = runtime_client.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='text/csv',
        Accept='text/csv',
        Body=payload,
        InferenceId=inference_id
    )
    response['Body'].read()  # drain the response

print(f'Sent {len(test_features)} requests')

It can take a few minutes for the data capture results to arrive in S3. Data Capture stops capturing requests at high levels of disk usage to prevent impact to inference requests. 

In [ ]:
import time as _time

s3_client = boto3.client('s3')
capture_prefix = f'{bucket_prefix}/data-capture/{endpoint_name}/AllTraffic'

print('Waiting for data capture files in S3 (may take 1-2 minutes)...')
for attempt in range(12):
    resp = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=capture_prefix, MaxKeys=1)
    if resp.get('KeyCount', 0) > 0:
        print(f'Data capture files found after {(attempt+1)*15}s')
        break
    _time.sleep(15)
else:
    print('No capture files found yet. They may still be buffering.')

# List captured files
resp = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=capture_prefix)
capture_keys = [obj['Key'] for obj in resp.get('Contents', [])]
print(f'\nFound {len(capture_keys)} capture file(s)')

Data capture stores the input and output in a JSONL file. This file can be loaded into a pandas DataFrame for calculating drift later.

In [ ]:
import base64

captured_records = []

for key in capture_keys:
    obj = s3_client.get_object(Bucket=bucket_name, Key=key)
    for line in obj['Body'].read().decode('utf-8').strip().split('\n'):
        record = json.loads(line)
        inference_id = record.get('eventMetadata', {}).get('inferenceId', '')

        input_data = record.get('captureData', {}).get('endpointInput', {})
        output_data = record.get('captureData', {}).get('endpointOutput', {})

        raw_input = input_data.get('data', '')
        if input_data.get('encoding') == 'BASE64':
            raw_input = base64.b64decode(raw_input).decode('utf-8')

        raw_output = output_data.get('data', '')
        if output_data.get('encoding') == 'BASE64':
            raw_output = base64.b64decode(raw_output).decode('utf-8')

        captured_records.append({
            'inference_id': inference_id,
            'input': raw_input.strip(),
            'prediction_proba': float(raw_output.strip()) if raw_output.strip() else None
        })

captured_df = pd.DataFrame(captured_records)
captured_df['prediction'] = (captured_df['prediction_proba'] > 0.5).astype(int)

feature_cols = pd.DataFrame(
    captured_df['input'].str.split(',').tolist(),
    columns=test_features.columns
).astype(float)
captured_df = pd.concat([captured_df[['inference_id', 'prediction_proba', 'prediction']], feature_cols], axis=1)

print(f'Parsed {len(captured_df)} captured records with inferenceId')
print(captured_df[['inference_id', 'prediction_proba', 'prediction']].head())

---

## 6: Data Drift Monitoring with Evidently

### What is Data Drift?

Data drift occurs when the statistical properties of input features change over time, which can degrade model performance.

### Why Monitor Drift?
- **Early Warning**: Detect issues before they impact business
- **Model Decay**: Understand when models need retraining
- **Data Quality**: Identify data pipeline issues
- **Compliance**: Track model performance for regulatory requirements

### Evidently Features
- **Open-source**: Free and customizable
- **Statistical Tests**: Multiple drift detection methods (KS test, Chi-square, etc.)
- **Visual Reports**: Interactive HTML reports
- **Flexible**: Works with any ML framework
- **MLflow Integration**: Log metrics and artifacts to MLflow

To calculate data drift, first load the baseline dataset saved during model training.

In [ ]:
reference_data = pd.read_csv('data/baseline.csv')

print(f'Reference (baseline) data shape: {reference_data.shape}')
print('\nReference data represents the training data distribution')
print('\nReference data statistics:')
print(reference_data.describe())

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
    <h3>⚠️ SIMULATED PRODUCTION DATA WITH ARTIFICIAL DRIFT</h3>
<p><strong>Purpose:</strong> Introducing controlled data drift to demonstrate drift detection capabilities</p>

<b>Important:</b> Introducing ARTIFICIAL data drift into the test set for demo purposes only. Do NOT use this in production. In a real scenario, this would be actual production data.

In [ ]:
feature_columns = [c for c in captured_df.columns if c not in ['inference_id', 'prediction_proba', 'prediction']]
production_data = captured_df[feature_columns].copy()

print(f'Production data shape: {production_data.shape}')
print('\nProduction data statistics:')
print(production_data.describe())
print('\nDrift was already injected into the payloads sent to the endpoint (see Section 5),')
print('so the Data Capture files — and therefore this DataFrame — already contain the shifted distribution.')


Evidently has various presets for calculating different types of [data drift](https://docs.evidentlyai.com/metrics/preset_data_drift), all of which can be customized to suit the particular ML use case. In addition to saving the final report in MLflow, it can be helpful to extract specific drift values and save them separately as metrics in MLflow. This enables you to easily compare different runs or generate alerts based on certain values. The cell below contains a helper function which extracts certain metrics from the Evidently results to log in MLflow. It can be customized depending on the types of data drift calculations you use.

In [ ]:
import numpy as np

def log_metrics_from_dict(d):
    metrics = d.get("metrics", []) or []

    drifted_columns_logged = False

    for m in metrics:
        metric_name = m.get("metric_name", "")
        cfg = m.get("config", {}) or {}
        val = m.get("value")

        # 1) DriftedColumnsCount: always log
        if metric_name.startswith("DriftedColumnsCount"):
            drifted_columns_logged = True
            # value is a dict: {"count": ..., "share": ...}
            if isinstance(val, dict):
                count = float(val.get("count", 0.0))
                share = float(val.get("share", 0.0))
            else:
                # Fallback if structure changes
                count = float(val) if val is not None else 0.0
                share = 0.0

            mlflow.log_metric("DriftedColumnsCount.count", count)
            mlflow.log_metric("DriftedColumnsCount.share", share)
            continue

        # 2) ValueDrift: log only if value > threshold
        if metric_name.startswith("ValueDrift"):
            threshold = cfg.get("threshold")
            column = cfg.get("column")

            if threshold is None or column is None:
                continue

            try:
                numeric_val = float(val)
            except (TypeError, ValueError):
                continue

            if numeric_val > float(threshold):
                key = f"ValueDrift:{column}"
                mlflow.log_metric(key, numeric_val)

    # 3) If there was no DriftedColumnsCount metric at all, log 0
    if not drifted_columns_logged:
        mlflow.log_metric("DriftedColumnsCount", 0.0)

In addition to extracting certain metrics, the entire reports produced by Evidently will be saved in a separate folder.

In [ ]:
os.makedirs('reports', exist_ok=True)

This sample uses the built-in `DataDriftPreset` and `DataSummaryPreset` from Evidently to calculate data drift from the baseline data and the inference results. The metrics are extracted and saved, along with the HTML and JSON reports, in the SageMaker MLflow App. 

In [ ]:
mlflow.set_experiment(mlflow_experiment_name)
timestamp_suffix = datetime.now().strftime('%Y%m%d_%H%M%S')

with mlflow.start_run(run_name=f"data_drift_quality_monitoring_{timestamp_suffix}") as drift_run:
    drift_run_id = drift_run.info.run_id
    
    mlflow.log_params({
        'reference_data_size': len(reference_data),
        'current_data_size': len(production_data),
        'monitoring_timestamp': datetime.now().isoformat(),
        'model_name': model_name,
        'model_package_arn': model_package_arn,
    })

    # Wrap DataFrames in Evidently Dataset objects
    drift_data_definition = DataDefinition()
    reference_dataset = Dataset.from_pandas(reference_data, data_definition=drift_data_definition)
    production_dataset = Dataset.from_pandas(production_data, data_definition=drift_data_definition)
    
    print('Creating Evidently data drift report...')
    data_drift_report = Report(metrics=[
        DataDriftPreset(),
    ])
    data_drift_report_snapshot = data_drift_report.run(
        reference_data=reference_dataset,
        current_data=production_dataset
    )
    
    data_drift_report_filename = f'data_drift_report_{timestamp_suffix}'
    data_drift_report_snapshot.save_html(f'reports/{data_drift_report_filename}.html')
    data_drift_report_snapshot.save_json(f'reports/{data_drift_report_filename}.json')
    
    mlflow.log_artifact(f'reports/{data_drift_report_filename}.html', 'evidently_report_data_drift_html')
    mlflow.log_artifact(f'reports/{data_drift_report_filename}.json', 'evidently_report_data_drift_json')
    
    drift_results = data_drift_report_snapshot.dict()
    log_metrics_from_dict(drift_results)

    print('Creating Evidently data summary report...')
    data_quality_report = Report(metrics=[
        DataSummaryPreset(),
    ])
    data_quality_report_snapshot = data_quality_report.run(
        reference_data=reference_data,
        current_data=production_data
    )
    
    data_quality_filename_html = f'data_quality_report_{timestamp_suffix}.html'
    data_quality_report_snapshot.save_html(f'reports/{data_quality_filename_html}')
    print(f'Data quality report saved to: {data_quality_filename_html}')
    mlflow.log_artifact(f'reports/{data_quality_filename_html}', 'evidently_report_data_quality')

The final report produced by Evidently can also be viewed directly inside this notebook.

In [ ]:
from IPython.display import IFrame

print('Displaying Evidently Data Drift Report:')
IFrame(src=f'reports/{data_drift_report_filename}.html', width=1000, height=800)

### View Drift Metrics in MLflow

To view your drift monitoring results in MLflow:

1. Open the SageMaker Studio MLflow UI
2. Navigate to the experiment: **demo-ml-xgb-train-monitor**
3. Find the run with name starting with **data_drift_**
4. View:
   - **Metrics**: drift scores for each feature, dataset-level drift
   - **Artifacts**: HTML reports and JSON summaries
   - **Parameters**: monitoring configuration and timestamps

---

## 7: Binary Classification Model Quality Evaluation with Evidently

Classification evaluation assesses how well your model performs on the prediction task by comparing predicted labels with actual ground truth labels.

### Key Classification Metrics

- **Accuracy**: Overall correctness of predictions
- **Precision**: Of all positive predictions, how many were actually positive (minimizes false positives)
- **Recall**: Of all actual positives, how many were correctly predicted (minimizes false negatives)
- **F1 Score**: Harmonic mean of precision and recall (balanced metric)
- **ROC-AUC**: Area under the ROC curve (measures discrimination ability)
- **Confusion Matrix**: Breakdown of true positives, true negatives, false positives, false negatives

### Why Evaluate Classification Performance?

- **Model Quality**: Understand model's predictive performance on test data
- **Business Metrics**: Align technical metrics with business objectives
- **Threshold Tuning**: Optimize decision threshold for your use case
- **Error Analysis**: Identify where the model makes mistakes

### Evidently Classification Features

- Comprehensive classification metrics (accuracy, precision, recall, F1, ROC-AUC)
- Confusion matrix visualization
- Classification quality by class
- Probability distribution analysis
- Prediction drift detection

First, merge the predictions produced by the real-time endpoint with the ground truth labels saved previously. These two datasets can be merged with the unique IDs.

In [ ]:
gt_df = pd.read_csv('data/ground_truth.csv')
classification_eval_data = captured_df.merge(gt_df, on='inference_id', how='inner')

print(f'Merged dataset shape: {classification_eval_data.shape}')
print(f'  Captured records: {len(captured_df)}')
print(f'  Ground truth records: {len(gt_df)}')
print(f'  Matched records: {len(classification_eval_data)}')
print(f'\nActual labels distribution:')
print(classification_eval_data['target'].value_counts())
print(f'\nPredicted labels distribution:')
print(classification_eval_data['prediction'].value_counts())

Similar to data drift, Evidently has various presets for calculating different types of [model quality](https://docs.evidentlyai.com/metrics/preset_classification), all of which can be customized to suit the particular ML use case. In addition to saving the final report in MLflow, it can be helpful to extract specific quality values and save them separately as metrics in MLflow. This enables you to easily compare different runs or generate alerts based on certain values. The cell below contains a helper function which extracts certain metrics from the Evidently results to log in MLflow. It can be customized depending on the types of quality calculations you use.

In [ ]:
import numpy as np

def log_evidently_classification_metrics_to_mlflow(metrics_dict):
    """Log Evidently classification metrics to MLflow using only metric prefix"""
    for metric in metrics_dict.get('metrics', []):
        # Extract only prefix before first '('
        metric_name_full = metric.get('metric_name', '')
        metric_name = metric_name_full.split('(')[0].strip()
        
        value = metric.get('value')
        
        if isinstance(value, dict):
            # Handle dict values like F1ByLabel, PrecisionByLabel, etc.
            for label, val in value.items():
                mlflow.log_metric(f"{metric_name}_label_{label}", float(val))
        else:
            # Handle scalar values (including np.float64)
            mlflow.log_metric(metric_name, float(value))

This sample uses the built-in `ClassificationPreset` from Evidently to calculate model quality from the inference results and the ground truth labels. The metrics are extracted and saved, along with the HTML and JSON reports, in the SageMaker MLflow App. 

In [ ]:
with mlflow.start_run(run_name=f'model_quality_{timestamp_suffix}') as run:
    eval_data = classification_eval_data.copy()
    print(f"Evaluation dataset columns: {eval_data.columns.tolist()}")
    print(f"Evaluation dataset shape: {eval_data.shape}")
    
    # Create DataDefinition with BinaryClassification to tell Evidently which columns to use
    # This is REQUIRED for ClassificationPreset to work
    data_definition = DataDefinition(
        classification=[
            BinaryClassification(
                target='target',              # Column with actual labels
                prediction_labels='prediction', # Column with predicted labels
                pos_label=1,                  # Positive class label
            )
        ],
    )
    
    # Wrap the DataFrame in an Evidently Dataset
    eval_dataset = Dataset.from_pandas(
        eval_data,
        data_definition=data_definition,
    )
    
    print('\nCreating Evidently classification performance report...')
    classification_report = Report(metrics=[
        ClassificationPreset(),
    ])
    classification_report_snapshot = classification_report.run(
        reference_data=None,
        current_data=eval_dataset,
    )
    
    classification_report_filename = f'classification_report_{timestamp_suffix}'
    classification_report_snapshot.save_html(f'data/{classification_report_filename}.html')
    print(f'Classification report saved to: data/{classification_report_filename}.html')
    
    classification_report_snapshot.save_json(f'data/{classification_report_filename}.json')
    print(f'Classification report saved to: data/{classification_report_filename}.json')
    
    mlflow.log_artifact(f'data/{classification_report_filename}.html', 'evidently_classification_report_html')
    mlflow.log_artifact(f'data/{classification_report_filename}.json', 'evidently_classification_report_json')
    
    classification_report_dict = classification_report_snapshot.dict()
    log_evidently_classification_metrics_to_mlflow(classification_report_dict)

### Classification Evaluation Summary

The classification report provides:

1. **Overall Performance Metrics**
   - Accuracy: Percentage of correct predictions
   - Precision: Quality of positive predictions
   - Recall: Coverage of actual positive cases
   - F1 Score: Balance between precision and recall
   - ROC-AUC: Discrimination ability across all thresholds

2. **Confusion Matrix**
   - True Positives (TP): Correctly predicted positive cases
   - True Negatives (TN): Correctly predicted negative cases
   - False Positives (FP): Incorrectly predicted as positive (Type I error)
   - False Negatives (FN): Incorrectly predicted as negative (Type II error)

3. **Class-Level Analysis**
   - Performance breakdown by class (0: No subscription, 1: Subscription)
   - Support: Number of samples per class
   - Per-class precision and recall

4. **Probability Calibration**
   - Distribution of predicted probabilities
   - Calibration curves showing reliability of probability estimates

### Business Impact

For the bank marketing use case:
- **High Precision**: Minimizes wasted effort on customers unlikely to subscribe
- **High Recall**: Maximizes capture of potential subscribers
- **F1 Score**: Balances both objectives for optimal campaign efficiency
- **Threshold Tuning**: Adjust decision boundary based on business costs/benefits

---

## 8: Automate Monitoring in a Scheduled Pipeline

The Evidently calculations above are great for ad-hoc analysis, but in production you want drift and model-quality checks to run automatically on a schedule, without opening a notebook. Wrap the Evidently logic in a **SageMaker Processing Job** orchestrated by a **SageMaker Pipeline** that:

- Pulls the latest Data Capture files from the endpoint and compares them to the training baseline
- Logs the Evidently reports and drift metrics to MLflow
- Publishes an **SNS** notification when the share of drifted features exceeds a threshold, so a human (or a downstream retraining pipeline) can react

![Model monitoring workflow with real-time endpoints](img/arch-sagemaker-inference-predictiveml-monitoring-RealTime-inf.png "Model monitoring workflow with real-time endpoints")

<div class="alert alert-info" role="alert">
<strong>Heads up:</strong> the pipeline execution role needs <code>sns:Publish</code> on the created topic to send alerts, and <code>scheduler:CreateSchedule</code> / <code>iam:PassRole</code> if you enable the EventBridge schedule. In workshop accounts some of these permissions may be missing — the code degrades gracefully (logs a warning, keeps the job successful).
</div>

### Create the SNS topic for drift alerts

The cell below creates an SNS topic and subscribes an email address. Email subscriptions require confirmation from the email owner. If the execution role does not have SNS permissions, the cell logs a warning and the pipeline still runs — it just won't publish the alert.

In [ ]:
ALERT_EMAIL = ''  # optional: set to receive drift alerts by email (must be confirmed)
SNS_TOPIC_NAME = 'sagemaker-evidently-drift-alerts'

sns_topic_arn = ''
try:
    sns_client = boto3.client('sns', region_name=region)
    sns_topic_arn = sns_client.create_topic(Name=SNS_TOPIC_NAME)['TopicArn']
    print(f'SNS topic ready: {sns_topic_arn}')
    if ALERT_EMAIL:
        sns_client.subscribe(TopicArn=sns_topic_arn, Protocol='email', Endpoint=ALERT_EMAIL)
        print(f'Subscription requested for {ALERT_EMAIL} — confirm via email to activate.')
except Exception as e:
    print(f'[WARN] Could not set up SNS ({e}). Pipeline will still run without alerts.')


### Write the Processing Job script

Materialize the Evidently monitoring script to `scripts/evidently_monitor.py` so the SageMaker Processing Job can reference it. Following the same `%%writefile` pattern used in earlier notebooks keeps the notebook self-contained.

In [ ]:
%%writefile scripts/evidently_monitor.py
import argparse
import base64
import glob
import json
import os
from datetime import datetime, timedelta, timezone

import boto3
import mlflow
import pandas as pd
from evidently import BinaryClassification, Dataset, DataDefinition, Report
from evidently.presets import ClassificationPreset, DataDriftPreset, DataSummaryPreset


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--mlflow-tracking-uri", required=True)
    p.add_argument("--mlflow-experiment", required=True)
    p.add_argument("--drift-threshold", type=float, default=0.05)
    p.add_argument("--lookback-days", type=int, default=7,
                   help="Only consider capture records within the last N days. 0 disables the filter.")
    p.add_argument("--critical-features", default="",
                   help="Comma-separated features that alert individually if drifted (e.g. 'campaign,age_18-29').")
    p.add_argument("--min-f1", type=float, default=0.0,
                   help="If >0, alert when ClassificationPreset F1 falls below this value.")
    p.add_argument("--sns-topic-arn", default="")
    p.add_argument("--endpoint-name", default="")
    return p.parse_args()


def load_capture(capture_dir, feature_names, lookback_days):
    cutoff = None
    if lookback_days and lookback_days > 0:
        cutoff = datetime.now(timezone.utc) - timedelta(days=lookback_days)
    rows, skipped = [], 0
    for path in glob.glob(os.path.join(capture_dir, "**", "*.jsonl"), recursive=True):
        with open(path) as f:
            for line in f:
                rec = json.loads(line)
                meta = rec.get("eventMetadata", {})
                if cutoff is not None:
                    ts_str = meta.get("inferenceTime", "")
                    try:
                        ts = datetime.fromisoformat(ts_str.replace("Z", "+00:00"))
                    except Exception:
                        ts = None
                    if ts is not None and ts < cutoff:
                        skipped += 1
                        continue
                cap = rec.get("captureData", {})
                inp, out = cap.get("endpointInput", {}), cap.get("endpointOutput", {})
                raw_in = inp.get("data", "")
                if inp.get("encoding") == "BASE64":
                    raw_in = base64.b64decode(raw_in).decode("utf-8")
                raw_out = out.get("data", "")
                if out.get("encoding") == "BASE64":
                    raw_out = base64.b64decode(raw_out).decode("utf-8")
                rows.append({
                    "inference_id": meta.get("inferenceId", ""),
                    "input": raw_in.strip(),
                    "proba": float(raw_out.strip()) if raw_out.strip() else None,
                })
    if skipped:
        print(f"Skipped {skipped} capture records older than {lookback_days} day(s).")
    if not rows:
        return None
    df = pd.DataFrame(rows)
    features = pd.DataFrame(df["input"].str.split(",").tolist(), columns=feature_names).astype(float)
    df["prediction"] = (df["proba"] > 0.5).astype(int)
    return pd.concat([df[["inference_id", "proba", "prediction"]], features], axis=1)


def drifted_share(report_dict):
    for m in report_dict.get("metrics", []):
        if m.get("metric_name", "").startswith("DriftedColumnsCount"):
            v = m.get("value")
            return float(v.get("share", 0.0)) if isinstance(v, dict) else 0.0
    return 0.0


def drifted_critical(report_dict, critical):
    """Return list of critical features whose ValueDrift exceeded their threshold."""
    hits = []
    if not critical:
        return hits
    for m in report_dict.get("metrics", []):
        if not m.get("metric_name", "").startswith("ValueDrift"):
            continue
        cfg = m.get("config", {}) or {}
        column, threshold, val = cfg.get("column"), cfg.get("threshold"), m.get("value")
        if column in critical and threshold is not None and val is not None:
            try:
                if float(val) > float(threshold):
                    hits.append(column)
            except (TypeError, ValueError):
                pass
    return hits


def all_drifted(report_dict):
    """Return list of (column, value) for every column whose ValueDrift exceeded its threshold, sorted desc."""
    hits = []
    for m in report_dict.get("metrics", []):
        if not m.get("metric_name", "").startswith("ValueDrift"):
            continue
        cfg = m.get("config", {}) or {}
        column, threshold, val = cfg.get("column"), cfg.get("threshold"), m.get("value")
        if column and threshold is not None and val is not None:
            try:
                fv = float(val)
                if fv > float(threshold):
                    hits.append((column, fv))
            except (TypeError, ValueError):
                pass
    return sorted(hits, key=lambda x: x[1], reverse=True)


def classification_f1(report_dict):
    for m in report_dict.get("metrics", []):
        if m.get("metric_name", "").startswith("F1Score") and not m.get("metric_name", "").startswith("F1ByLabel"):
            try:
                return float(m.get("value"))
            except (TypeError, ValueError):
                return None
    return None


def publish_alert(topic_arn, subject, message):
    if not topic_arn:
        return
    try:
        # Extract region from the SNS topic ARN (arn:aws:sns:<region>:<acct>:<name>)
        region = topic_arn.split(":")[3] if topic_arn.count(":") >= 3 else os.environ.get("AWS_REGION", "us-east-1")
        boto3.client("sns", region_name=region).publish(TopicArn=topic_arn, Subject=subject, Message=message)
        print(f"SNS alert published: {subject}")
    except Exception as e:
        print(f"[WARN] SNS publish skipped: {e}")


def main():
    args = parse_args()
    base_in, base_out = "/opt/ml/processing/input", "/opt/ml/processing/output"
    os.makedirs(base_out, exist_ok=True)

    reference = pd.read_csv(f"{base_in}/baseline/baseline.csv")
    captured = load_capture(f"{base_in}/capture", feature_names=reference.columns.tolist(),
                            lookback_days=args.lookback_days)
    if captured is None or len(captured) == 0:
        print(f"[WARN] No capture records found within the last {args.lookback_days} day(s). "
              f"Nothing to compare — exiting successfully.")
        return

    production = captured[reference.columns.tolist()].copy()
    critical = [c.strip() for c in args.critical_features.split(",") if c.strip()]

    mlflow.set_tracking_uri(args.mlflow_tracking_uri)
    mlflow.set_experiment(args.mlflow_experiment)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")

    with mlflow.start_run(run_name=f"scheduled_monitoring_{ts}"):
        mlflow.log_params({
            "reference_size": len(reference),
            "current_size": len(production),
            "drift_threshold": args.drift_threshold,
            "lookback_days": args.lookback_days,
            "critical_features": args.critical_features,
            "min_f1": args.min_f1,
            "endpoint_name": args.endpoint_name,
        })

        snap = Report(metrics=[DataDriftPreset(), DataSummaryPreset()]).run(
            reference_data=Dataset.from_pandas(reference, data_definition=DataDefinition()),
            current_data=Dataset.from_pandas(production, data_definition=DataDefinition()),
        )
        html_path, json_path = f"{base_out}/data_drift_{ts}.html", f"{base_out}/data_drift_{ts}.json"
        snap.save_html(html_path)
        snap.save_json(json_path)
        mlflow.log_artifact(html_path, "evidently_report_data_drift_html")
        mlflow.log_artifact(json_path, "evidently_report_data_drift_json")

        drift_dict = snap.dict()
        share = drifted_share(drift_dict)
        mlflow.log_metric("DriftedColumnsCount.share", share)
        print(f"Drifted columns share: {share:.3f} (threshold={args.drift_threshold})")

        all_hits = all_drifted(drift_dict)
        if all_hits:
            print(f"All drifted features: {all_hits}")

        critical_hits = drifted_critical(drift_dict, critical)
        if critical_hits:
            print(f"Critical features drifted: {critical_hits}")

        f1 = None
        gt_path = f"{base_in}/ground_truth/ground_truth.csv"
        if os.path.exists(gt_path):
            gt = pd.read_csv(gt_path)
            eval_df = captured.merge(gt, on="inference_id", how="inner")
            if len(eval_df):
                dd = DataDefinition(classification=[
                    BinaryClassification(target="target", prediction_labels="prediction", pos_label=1)
                ])
                cls_snap = Report(metrics=[ClassificationPreset()]).run(
                    reference_data=None,
                    current_data=Dataset.from_pandas(eval_df, data_definition=dd),
                )
                cls_html = f"{base_out}/classification_{ts}.html"
                cls_snap.save_html(cls_html)
                mlflow.log_artifact(cls_html, "evidently_classification_report_html")
                f1 = classification_f1(cls_snap.dict())
                if f1 is not None:
                    mlflow.log_metric("F1Score", f1)
                    print(f"Classification F1: {f1:.3f} (min_f1={args.min_f1})")

        # Consolidate triggers
        reasons = []
        if share > args.drift_threshold:
            reasons.append(f"drifted columns share {share:.2%} > threshold {args.drift_threshold:.2%}")
        if critical_hits:
            reasons.append(f"critical features drifted: {', '.join(critical_hits)}")
        if args.min_f1 > 0 and f1 is not None and f1 < args.min_f1:
            reasons.append(f"F1 {f1:.3f} < min_f1 {args.min_f1:.3f}")

        if reasons:
            run = mlflow.active_run()
            run_id = run.info.run_id if run else "n/a"
            experiment_id = run.info.experiment_id if run else "n/a"
            mlflow_ui_hint = (
                f"Open SageMaker Studio -> MLflow -> Experiment '{args.mlflow_experiment}' "
                f"-> run_id {run_id}"
            )
            drifted_block = ""
            if all_hits:
                drifted_block = "Drifted features (feature: score):\n  - " + "\n  - ".join(
                    f"{col}: {score:.4f}" for col, score in all_hits
                ) + "\n\n"

            publish_alert(
                args.sns_topic_arn,
                subject=f"[SageMaker] Model monitoring alert — {args.endpoint_name}",
                message=(
                    f"Endpoint: {args.endpoint_name}\n"
                    f"Run timestamp: {ts}\n\n"
                    f"Triggers:\n  - " + "\n  - ".join(reasons) + "\n\n"
                    + drifted_block +
                    f"MLflow experiment: {args.mlflow_experiment} (id {experiment_id})\n"
                    f"MLflow run_id:     {run_id}\n"
                    f"{mlflow_ui_hint}\n"
                ),
            )


if __name__ == "__main__":
    main()


### Define the monitoring pipeline

Single `ProcessingStep` running `scripts/evidently_monitor.py` on a `FrameworkProcessor`. Pipeline parameters let you tune each execution without redeploying:

- `DriftThreshold` — share of drifted columns that triggers an SNS alert
- `LookbackDays` — only consider capture records within the last N days (set to `0` to disable and compare against the full history)
- `CriticalFeatures` — comma-separated features that raise an alert individually if they drift, even when the overall share is below the threshold
- `MinF1` — when > 0, also alert when classification F1 falls below this value (requires ground-truth)
- `SnsTopicArn`, `EndpointName` — wired to the SNS topic and endpoint created earlier

The SageMaker-managed scikit-learn image doesn't ship with Evidently or MLflow. A `requirements.txt` next to the script is auto-installed by `FrameworkProcessor` before running.

In [ ]:
%%writefile scripts/requirements.txt
evidently==0.7.17
pandas>2,<3
mlflow<4
sagemaker-mlflow


In [ ]:
from sagemaker.mlops.workflow.pipeline import Pipeline
from sagemaker.mlops.workflow.steps import ProcessingStep, CacheConfig
from sagemaker.core.workflow.parameters import ParameterFloat, ParameterString, ParameterInteger
from sagemaker.core.workflow.pipeline_context import PipelineSession
from sagemaker.core.processing import FrameworkProcessor
from sagemaker.core.shapes import ProcessingInput, ProcessingS3Input, ProcessingOutput, ProcessingS3Output
from sagemaker.core import image_uris

monitoring_pipeline_name = 'evidently-monitoring-pipeline'

p_drift_threshold = ParameterFloat(name='DriftThreshold', default_value=0.05)
p_sns_topic = ParameterString(name='SnsTopicArn', default_value=sns_topic_arn or '')
p_endpoint = ParameterString(name='EndpointName', default_value=endpoint_name)
p_lookback_days = ParameterInteger(name='LookbackDays', default_value=7)
p_critical_features = ParameterString(name='CriticalFeatures', default_value='campaign,age_18-29')
p_min_f1 = ParameterFloat(name='MinF1', default_value=0.0)

endpoint_capture_s3 = f'{capture_s3_uri}/{endpoint_name}/AllTraffic'

ground_truth_s3 = sagemaker_session.upload_data(
    'data/ground_truth.csv', bucket_name, f'{bucket_prefix}/data/ground-truth'
)

processor_image_uri = image_uris.retrieve(framework='sklearn', version='1.4-2', region=region)

monitor_processor = FrameworkProcessor(
    image_uri=processor_image_uri,
    role=sm_role,
    instance_type='ml.m5.large',
    instance_count=1,
    sagemaker_session=PipelineSession(),
    base_job_name='evidently-monitor',
    command=['python3'],
)

monitor_inputs = [
    ProcessingInput(input_name='baseline', s3_input=ProcessingS3Input(
        s3_uri=baseline_s3, local_path='/opt/ml/processing/input/baseline', s3_data_type='S3Prefix')),
    ProcessingInput(input_name='capture', s3_input=ProcessingS3Input(
        s3_uri=endpoint_capture_s3, local_path='/opt/ml/processing/input/capture', s3_data_type='S3Prefix')),
    ProcessingInput(input_name='ground_truth', s3_input=ProcessingS3Input(
        s3_uri=ground_truth_s3, local_path='/opt/ml/processing/input/ground_truth', s3_data_type='S3Prefix')),
]

monitor_outputs = [
    ProcessingOutput(output_name='reports', s3_output=ProcessingS3Output(
        s3_uri=f's3://{bucket_name}/{bucket_prefix}/evidently-reports',
        local_path='/opt/ml/processing/output',
        s3_upload_mode='EndOfJob')),
]

monitor_step = ProcessingStep(
    name='EvidentlyMonitoring',
    step_args=monitor_processor.run(
        code='evidently_monitor.py',
        source_dir='scripts',
        inputs=monitor_inputs,
        outputs=monitor_outputs,
        arguments=[
            '--mlflow-tracking-uri', mlflow_arn,
            '--mlflow-experiment', mlflow_experiment_name,
            '--drift-threshold', p_drift_threshold.to_string(),
            '--lookback-days', p_lookback_days.to_string(),
            '--critical-features', p_critical_features,
            '--min-f1', p_min_f1.to_string(),
            '--sns-topic-arn', p_sns_topic,
            '--endpoint-name', p_endpoint,
        ],
    ),
    cache_config=CacheConfig(enable_caching=False),
)

monitoring_pipeline = Pipeline(
    name=monitoring_pipeline_name,
    parameters=[p_drift_threshold, p_sns_topic, p_endpoint, p_lookback_days, p_critical_features, p_min_f1],
    steps=[monitor_step],
)
print(f'Pipeline defined: {monitoring_pipeline_name}')

### Upsert the pipeline and (optionally) attach a schedule

`pipeline.upsert()` creates or updates the pipeline definition. The optional schedule is attached via **Amazon EventBridge Scheduler**, which natively supports `StartPipelineExecution` as a target. Set `ENABLE_SCHEDULE = True` to activate. Default is a daily run at 08:00 UTC.

In [ ]:
import json as _json

ENABLE_SCHEDULE = True  # flip to True to activate the daily run
SCHEDULE_EXPRESSION = 'cron(0 8 * * ? *)'  # every day at 08:00 UTC
# SCHEDULE_EXPRESSION = 'cron(0/20 * * * ? *)' # every 20 minutes
SCHEDULE_NAME = f'{monitoring_pipeline_name}-daily'
# SCHEDULE_NAME = f'{monitoring_pipeline_name}-20mins'

monitoring_pipeline.upsert(role_arn=sm_role)
print(f'Pipeline upserted: {monitoring_pipeline.name}')

if ENABLE_SCHEDULE:
    try:
        scheduler = boto3.client('scheduler', region_name=region)
        pipeline_arn = f'arn:aws:sagemaker:{region}:{boto3.client("sts").get_caller_identity()["Account"]}:pipeline/{monitoring_pipeline.name}'
        scheduler.create_schedule(
            Name=SCHEDULE_NAME,
            ScheduleExpression=SCHEDULE_EXPRESSION,
            FlexibleTimeWindow={'Mode': 'OFF'},
            Target={
                'Arn': pipeline_arn,
                'RoleArn': sm_role,
                'SageMakerPipelineParameters': {
                    'PipelineParameterList': [
                        {'Name': 'DriftThreshold', 'Value': '0.05'},
                        {'Name': 'LookbackDays', 'Value': '7'},
                        {'Name': 'CriticalFeatures', 'Value': 'campaign,age_18-29'},
                        {'Name': 'MinF1', 'Value': '0.0'},
                        {'Name': 'SnsTopicArn', 'Value': sns_topic_arn or ''},
                        {'Name': 'EndpointName', 'Value': endpoint_name},
                    ]
                },
            }
        )
        print(f'Schedule created: {SCHEDULE_NAME} ({SCHEDULE_EXPRESSION})')
    except Exception as e:
        print(f'[WARN] Could not create schedule ({e}). Start the pipeline manually below.')
else:
    print('Schedule disabled (ENABLE_SCHEDULE=False). Start the pipeline manually below.')

### Run the pipeline once manually

Kick off an execution now to validate the end-to-end flow. The cell below starts the pipeline, renders a link to the execution graph in SageMaker Studio, waits for completion, and lists the steps. After that, it prints the MLflow run ID so you can jump straight to the Evidently artifacts in the MLflow UI.

> **Heads up:** Data Capture buffers requests in S3 and can take 1–2 minutes to flush after the endpoint invocations. If the job fails with `No capture records found`, wait a minute and re-run the cell.

In [ ]:
from IPython.display import HTML, display

execution = monitoring_pipeline.start(
    parameters={
        'DriftThreshold': 0.05,
        'LookbackDays': 7,              # compare only capture from the last 7 days; set 0 to disable
        'CriticalFeatures': 'campaign,age_18-29',  # alert individually if any of these drift
        'MinF1': 0.0,                   # set > 0 to also alert on F1 degradation
        'SnsTopicArn': sns_topic_arn or '',
        'EndpointName': endpoint_name,
    }
)
execution_id = execution.describe()['PipelineExecutionArn'].split('/')[-1]
print(f'Pipeline execution started: {execution_id}')

display(HTML(
    f'<b>See <a target="_blank" href="https://studio-{domain_id}.studio.{region}.sagemaker.aws/'
    f'pipelines/{monitoring_pipeline_name}/executions/{execution_id}/graph">the pipeline execution</a> in the Studio UI</b>'
))

%time execution.wait()
execution.list_steps()

In [ ]:
# Show the MLflow run produced by the scheduled monitoring job
from mlflow import MlflowClient

mlflow.set_tracking_uri(mlflow_arn)
client = MlflowClient()
exp = client.get_experiment_by_name(mlflow_experiment_name)
runs = client.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string="tags.mlflow.runName LIKE 'scheduled_monitoring_%'",
    order_by=['start_time DESC'],
    max_results=1,
)
if runs:
    r = runs[0]
    print(f"MLflow run: {r.info.run_name}")
    print(f"  run_id:    {r.info.run_id}")
    print(f"  share:     {r.data.metrics.get('DriftedColumnsCount.share', 'n/a')}")
else:
    print('No monitoring run found yet — check the Studio pipeline execution logs.')


---

## 9: Comprehensive Monitoring Report (Optional)

Create a combined report with drift detection and data quality analysis.

In [ ]:
with mlflow.start_run(run_name=f"comprehensive_monitoring_{timestamp_suffix}") as comp_run:
    print('Creating comprehensive monitoring report...')
    comprehensive_report = Report(metrics=[
        DataDriftPreset(),
        DataSummaryPreset(),
    ])
    comprehensive_report_snapshot = comprehensive_report.run(
        reference_data=reference_data,
        current_data=production_data
    )
    
    comp_filename = f'comprehensive_monitoring_{datetime.now().strftime("%Y%m%d_%H%M%S")}.html'
    comprehensive_report_snapshot.save_html(comp_filename)
    print(f'Comprehensive report saved to: {comp_filename}')

    mlflow.log_artifact(comp_filename, 'evidently_comprehensive_report')
    comp_s3_key = f"{bucket_prefix}/evidently-reports/{comp_filename}"
    s3_client.upload_file(comp_filename, bucket_name, comp_s3_key)
    mlflow.log_param('comprehensive_report_s3', f's3://{bucket_name}/{comp_s3_key}')
    print(f'\nComprehensive monitoring report completed')
    print(f'Report location: s3://{bucket_name}/{comp_s3_key}')

---

## 10: Summary and Next Steps

### What We Accomplished

1. **Model Registry**: Selected the latest approved model version produced by the SageMaker pipeline, made sure it was approved, and validated its serving code locally
2. **Experiment Tracking**: Logged all parameters, metrics, and artifacts to SageMaker AI MLflow
3. **Real-Time Inference**: Deployed the approved Model Package version to a SageMaker real-time endpoint with data capture enabled, using the typed `Model` / `EndpointConfig` / `Endpoint` resources
4. **Data Capture**: Automatically captured request/response payloads to S3 for monitoring
5. **Classification Evaluation**: Evaluated binary classification performance using Evidently's ClassificationPreset with comprehensive metrics (accuracy, precision, recall, F1, ROC-AUC, confusion matrix)
6. **Drift Detection**: Used Evidently to detect data drift between training baseline and captured production data
7. **MLflow Integration**: Logged all classification metrics, drift metrics, and reports to MLflow for tracking and comparison
8. **Visual Reports**: Generated interactive HTML reports for classification performance, data drift, and quality analysis
9. **Scheduled Monitoring Pipeline**: Wrapped the Evidently checks in a SageMaker Processing Job orchestrated by a SageMaker Pipeline, with parameters for drift threshold, lookback window, critical features, and minimum F1 — optionally triggered daily via EventBridge Scheduler
10. **Drift Alerting**: Published SNS notifications when the share of drifted features, a critical feature, or the classification F1 crosses the configured thresholds

### Next Steps

1. **Event-Driven Monitoring**: Trigger the monitoring pipeline from an **S3 event** on the data-capture prefix (via EventBridge rule → `StartPipelineExecution`) instead of (or alongside) the time-based schedule, so drift checks run as soon as fresh capture records land
2. **Closed-Loop Retraining**: Extend the pipeline with a conditional step that starts a retraining pipeline when drift or F1 breaches the thresholds, promoting the new model through the MLflow model registry and redeploying the endpoint
3. **Enhanced Monitoring**: Track prediction-distribution drift and business KPIs (e.g., subscription rate, revenue impact) over time in addition to feature drift
4. **Dashboards & Trends**: Build drift and quality trend visualizations on top of the MLflow metrics (e.g., an MLflow `compare_runs` view, a QuickSight dashboard over the logged metrics, or a custom Streamlit/Studio app)
5. **Robustness & Cost**: Tune lookback window and sampling rate, turn on pipeline step caching for idempotent reruns, and add retries / alerts on pipeline failures themselves (not only on drift)

---

## 11: Cleanup (Optional)

Clean up resources to avoid unnecessary costs. **Important**: Real-time endpoints incur charges while running.

In [ ]:
sm_client = boto3.client('sagemaker')

try:
    sm_client.delete_endpoint(EndpointName=endpoint_name)
    print(f'Deleting endpoint: {endpoint_name} (this may take a minute)')
except Exception as e:
    print(f'Error deleting endpoint: {e}')

try:
    sm_client.delete_endpoint_config(EndpointConfigName=endpoint_name)
    print(f'Deleted endpoint config')
except Exception as e:
    print(f'Error deleting endpoint config: {e}')

try:
    sm_client.delete_model(ModelName=model_name)
    print(f'Deleted model: {model_name}')
except Exception as e:
    print(f'Error deleting model: {e}')

try:
    boto3.client('scheduler', region_name=region).delete_schedule(Name=SCHEDULE_NAME)
    print(f'Deleted schedule: {SCHEDULE_NAME}')
except Exception as e:
    print(f'No schedule to delete or error: {e}')

try:
    monitoring_pipeline.delete()
    print(f'Deleted pipeline: {monitoring_pipeline.name}')
except Exception as e:
    print(f'Error deleting pipeline: {e}')

try:
    if sns_topic_arn:
        sns_client.delete_topic(TopicArn=sns_topic_arn)
        print(f'Deleted SNS topic: {sns_topic_arn}')
except Exception as e:
    print(f'Error deleting SNS topic: {e}')

try:
    reports_prefix = f'{bucket_prefix}/evidently-reports'
    objects = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=reports_prefix).get('Contents', [])
    if objects:
        s3_client.delete_objects(Bucket=bucket_name, Delete={'Objects': [{'Key': o['Key']} for o in objects]})
        print(f'Deleted {len(objects)} Evidently report objects from s3://{bucket_name}/{reports_prefix}')
except Exception as e:
    print(f'Error deleting Evidently reports: {e}')

print('\nNote: MLflow runs and S3 data are preserved for historical analysis.')

---

## Additional Resources

### Documentation
- [SageMaker Python SDK v3](https://sagemaker.readthedocs.io/en/stable/)
- [SageMaker AI MLflow](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html)
- [Evidently Documentation](https://docs.evidentlyai.com/)

### GitHub Repositories
- [Evidently GitHub](https://github.com/evidentlyai/evidently)
- [SageMaker Examples](https://github.com/aws/amazon-sagemaker-examples)

## Continue with the step 5
In the next notebook [05-sagemaker-project](05-sagemaker-project.ipynb) you automate model building with a CI/CD pipeline using SageMaker Projects. The step 6 notebook then adds the model deployment pipeline.